# End-to-End CLV Pipeline

This notebook represents the complete, unified workflow. It combines both the data generation steps and the machine learning training steps into one seamless execution.

**Run this notebook from top to bottom before starting the Streamlit application.** It guarantees that all datasets (`customers.csv`, `customers_clv.csv`) and model artifacts (`scaler.pkl`, `rf_model.pkl`, etc.) are fresh and correctly populated.

## Step 1: Import All Necessary Libraries
We import all libraries required for both data engineering and machine learning.

In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
import shap

np.random.seed(42)

## Step 2: Define Data Generation Logic
We define the helper function to create segmented synthetic customer profiles based on configurable parameters. We use realistic retail assumptions, such as a 35% gross margin.

In [ ]:
N = 3000
GROSS_MARGIN = 0.35

def generate_tier(n, tier_name, config):
    data = {
        'tenure_months':     np.random.randint(config['tenure'][0], config['tenure'][1], n),
        'total_orders':      np.random.randint(config['orders'][0], config['orders'][1], n),
        'total_spend':       np.clip(
            np.random.lognormal(mean=np.log(config['spend_mean']), sigma=config['spend_sigma'], size=n),
            config['spend_clip'][0], config['spend_clip'][1]
        ),
        'recency_days':      np.random.randint(config['recency'][0], config['recency'][1], n),
        'age':               np.random.randint(config['age'][0], config['age'][1], n),
        'income_bracket':    np.random.choice(config['income'], size=n),
        'nps_score':         np.random.randint(config['nps'][0], config['nps'][1], n),
        'online_ratio':      np.random.uniform(config['online'][0], config['online'][1], n),
        'return_rate':       np.random.uniform(config['return_rate'][0], config['return_rate'][1], n),
        'support_tickets':   np.random.randint(config['tickets'][0], config['tickets'][1], n),
        'discount_usage':    np.random.uniform(config['discount'][0], config['discount'][1], n),
        'region':            np.random.choice(['North', 'South', 'East', 'West'], size=n),
        'gender':            np.random.choice(['M', 'F'], size=n),
        'clv_tier':          tier_name,
    }
    return pd.DataFrame(data)

## Step 3: Generate the Customer Base
We specify the configurations for Champions, Growers, At-Risk, and Hibernating customers, generate them, and merge them into one dataset.

In [ ]:
tier1_config = {'tenure': (24, 84), 'orders': (40, 200), 'spend_mean': 8000, 'spend_sigma': 0.4, 'spend_clip': (2500, 15000), 'recency': (1, 30), 'age': (30, 65), 'income': [3, 4, 4, 4], 'nps': (8, 11), 'online': (0.3, 0.8), 'return_rate': (0.01, 0.08), 'tickets': (0, 5), 'discount': (0.0, 0.15)}
tier2_config = {'tenure': (6, 36), 'orders': (12, 60), 'spend_mean': 1500, 'spend_sigma': 0.45, 'spend_clip': (800, 2500), 'recency': (15, 90), 'age': (25, 55), 'income': [2, 3, 3], 'nps': (6, 10), 'online': (0.4, 0.9), 'return_rate': (0.05, 0.15), 'tickets': (1, 8), 'discount': (0.1, 0.35)}
tier3_config = {'tenure': (12, 48), 'orders': (4, 20), 'spend_mean': 450, 'spend_sigma': 0.5, 'spend_clip': (200, 800), 'recency': (60, 200), 'age': (20, 70), 'income': [1, 2, 2], 'nps': (4, 8), 'online': (0.2, 0.6), 'return_rate': (0.1, 0.3), 'tickets': (3, 15), 'discount': (0.3, 0.7)}
tier4_config = {'tenure': (1, 24), 'orders': (1, 8), 'spend_mean': 80, 'spend_sigma': 0.6, 'spend_clip': (10, 200), 'recency': (180, 365), 'age': (18, 75), 'income': [1, 1, 2], 'nps': (1, 7), 'online': (0.1, 0.5), 'return_rate': (0.2, 0.5), 'tickets': (2, 20), 'discount': (0.4, 1.0)}

df = pd.concat([
    generate_tier(600, 'Champions', tier1_config),
    generate_tier(900, 'Growers', tier2_config),
    generate_tier(800, 'At-Risk', tier3_config),
    generate_tier(700, 'Hibernating', tier4_config)
], ignore_index=True)

df['customer_id'] = [f"CUST_{i+1:05d}" for i in range(len(df))]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Step 4: Feature Engineering and Target Formulation
Compute foundational RFM (Recency, Frequency, Monetary) quintiles, estimate retention probabilities, and construct the final Target Variable (`clv_12month`).

In [ ]:
df['avg_order_value'] = (df['total_spend'] / df['total_orders']).clip(lower=1.0)
df['tenure_years'] = df['tenure_months'] / 12.0
df['purchase_frequency'] = (df['total_orders'] / df['tenure_years']).clip(upper=365.0)

df['recency_score'] = pd.qcut(df['recency_days'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)
df['frequency_score'] = pd.qcut(df['total_orders'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
df['monetary_score'] = pd.qcut(df['total_spend'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
df['rfm_score'] = df['recency_score'] + df['frequency_score'] + df['monetary_score']

df['retention_rate'] = ((df['recency_score'] + df['frequency_score']) / 10.0 + np.random.normal(0, 0.05, len(df))).clip(0.05, 0.95)
df['churn_rate'] = 1.0 - df['retention_rate']

raw_clv = (df['avg_order_value'] * df['purchase_frequency'] * GROSS_MARGIN) / df['churn_rate']
df['clv_12month'] = (raw_clv * np.random.lognormal(0, 0.15, len(df))).clip(5.0, 50000.0).round(2)
df['clv_segment'] = pd.cut(df['clv_12month'], bins=[0, 200, 800, 2500, 50001], labels=['Low', 'Medium', 'High', 'Very High'])

os.makedirs("../data", exist_ok=True)
df.to_csv("../data/customers.csv", index=False)
print("Data Generation Complete.")

## Step 5: Machine Learning Preparation
Select the features for the model, log-transform the target to handle right-skewness, and scale the features using Standardization.

In [ ]:
FEATURES = ['tenure_months', 'total_orders', 'total_spend', 'recency_days', 'age', 'income_bracket', 'nps_score', 'online_ratio', 'return_rate', 'support_tickets', 'discount_usage', 'avg_order_value', 'tenure_years', 'purchase_frequency', 'recency_score', 'frequency_score', 'monetary_score', 'rfm_score', 'retention_rate', 'churn_rate']
X = df[FEATURES].copy()
y_log = np.log1p(df['clv_12month'])

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")

## Step 6: Model Training
Train Ridge Regression, Random Forest, and XGBoost models concurrently. The Random Forest acts as our primary model, XGBoost pushes for maximum accuracy, and Ridge provides a linear baseline.

In [ ]:
ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
joblib.dump(ridge, "../models/ridge_model.pkl")

rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_split=10, min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=42)
rf.fit(X_train_scaled, y_train)
joblib.dump(rf, "../models/rf_model.pkl")

xgb = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, eval_metric='rmse')
xgb.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)
joblib.dump(xgb, "../models/xgb_model.pkl")

## Step 7: Calculate SHAP Values and Evaluate Metrics
Generate SHAP components for the dashboard to render local explanations. We also evaluate the Root Mean Squared Error and R-squared for all algorithms on the test set.

In [ ]:
explainer = shap.TreeExplainer(rf)
joblib.dump(explainer, "../models/shap_explainer.pkl")

y_test_dollars = np.expm1(y_test)
def eval_model(y_pred_log):
    y_pred = np.expm1(y_pred_log)
    return {'MAE': mean_absolute_error(y_test_dollars, y_pred), 'RMSE': np.sqrt(mean_squared_error(y_test_dollars, y_pred)), 'R2': r2_score(y_test_dollars, y_pred), 'MAPE': mean_absolute_percentage_error(y_test_dollars, y_pred) * 100}

metrics = {
    'Ridge': eval_model(ridge.predict(X_test_scaled)),
    'Random Forest': eval_model(rf.predict(X_test_scaled)),
    'XGBoost': eval_model(xgb.predict(X_test_scaled)),
    'Ensemble': eval_model(0.5*rf.predict(X_test_scaled) + 0.3*xgb.predict(X_test_scaled) + 0.2*ridge.predict(X_test_scaled))
}
joblib.dump(metrics, "../models/metrics.pkl")
for m, vals in metrics.items():
    print(f"{m:15}: R²={vals['R2']:.4f}, MAE=${vals['MAE']:,.2f}")

## Step 8: Apply Predictions to Full Dataset
We push the entire customer base through all models to get their predicted CLVs, which we save for direct ingestion by the Streamlit dashboard.

In [ ]:
X_all_scaled = scaler.transform(df[FEATURES])
df['clv_predicted_rf'] = np.expm1(rf.predict(X_all_scaled))
df['clv_predicted_ridge'] = np.expm1(ridge.predict(X_all_scaled))
df['clv_predicted_xgb'] = np.expm1(xgb.predict(X_all_scaled))
df['clv_predicted_ensemble'] = np.expm1(0.5 * df['clv_predicted_rf'] + 0.3 * df['clv_predicted_xgb'] + 0.2 * df['clv_predicted_ridge'])

df.to_csv("../data/customers_clv.csv", index=False)
print("Pipeline complete. All artifacts saved successfully.")